In [ ]:
import pickle
import matplotlib.pyplot as plt
import cv2
import os

In [ ]:
'''
    This is a script that you can use to see how your model predicts 
    keypoints on our test dataset.

    After running test.py metrics are output to the console and a .pkl 
    file is saved (dump file).

    Set the path to the folder with the test dataset, your dump (.pkl) 
    file, and the path to the folder where the marked-up snapshots will 
    be saved.

    dump_file:   path to the .pkl file
    image_path:  path to the test dataset
    output_path: path to save snapshots
'''

dump_file = "dump.pkl"  # Model results after running the test script
image_path = "test_images/"  # Folder with test images
output_path = "marked_images/"  # Specify the folder where the marked-up images will be saved

# Load model prediction
with open(dump_file, 'rb') as file:
    model_prediction = pickle.load(file)
file.close()

# Main loop
images = os.listdir(image_path)

for im in images:
    image_name = im
    for i in range(len(model_prediction)):
        if os.path.basename(model_prediction[i]["img_path"]) == image_name:
            mask = model_prediction[i]["gt_instances"]["keypoints_visible"][0]

            pred_x, pred_y = zip(*model_prediction[i]["pred_instances"]["keypoints"][0])
            gt_x, gt_y = zip(*model_prediction[i]["gt_instances"]["keypoints"][0])

            filtered_pred_x = [point for point, m in zip(pred_x, mask) if m == 1.]
            filtered_pred_y = [point for point, m in zip(pred_y, mask) if m == 1.]
            filtered_gt_x = [point for point, m in zip(gt_x, mask) if m == 1.]
            filtered_gt_y = [point for point, m in zip(gt_y, mask) if m == 1.]

            img = cv2.imread(image_path + image_name)

            # Build and save marked image
            plt.figure(figsize = (8, 12))
            plt.imshow(img)
            plt.scatter(gt_x, gt_y, s = 10, color = "yellow", edgecolors="black", linewidths = 0.5, label = "Grand True")
            plt.scatter(filtered_pred_x, filtered_pred_y, s = 10, color = "red", edgecolors="black", linewidths = 0.5, label = "Prediction")
            plt.legend()
            plt.axis('off')
            plt.savefig(output_path + image_name, dpi = 300, bbox_inches = "tight", pad_inches = 0)
            plt.close()

            break